## Implementing a Text Generator Using Long Short-Term Memory (LSTM) Networks
In this section, we build the same character-based text generator as the RNN notebook, but swap the recurrent layer for an **LSTM** (Long Short-Term Memory) in PyTorch. An LSTM is still a recurrent network — it still walks through the sequence one character at a time — but each cell carries an extra **cell state** alongside the hidden state, and uses three gates (forget / input / output) to control what gets kept, updated, and exposed. This lets it retain information over longer sequences without the vanishing-gradient problems that limit vanilla RNNs.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

### <br>2. Defining the Input Text and Prepare Character Set
We define the input text and identify unique characters in the text which we'll encode for our model.

In [ ]:
text = """In the ancient kingdom of Arandor there lived a wise king named Aldric. 
Every spring the king traveled from the northern mountains to the southern coast to visit the people of his kingdom. 
During these journeys he carried a silver key that opened a hidden chamber beneath the royal castle. 
Only the king knew the purpose of the key.
One year a young scholar named Elian joined the royal court. 
Elian was curious about many things, but he was especially curious about the silver key.
Whenever the king returned from his travels, Elian asked questions about the hidden chamber.
The king always smiled and changed the subject. Months passed. 
The kingdom celebrated festivals, harvested crops, and welcomed merchants from distant lands.
Throughout all these events the king continued to carry the silver key wherever he went.
Many people noticed the key, but nobody knew its purpose.
One winter a great storm struck the kingdom.
Roads disappeared beneath snow and communication between villages became difficult.
The king gathered his advisors and discussed how to help the people.
During the meeting the silver key rested on the table beside him.
After the storm ended, Elian once again asked about the silver key.
The king smiled but did not answer.
Instead he told Elian that patience was one of the most important virtues a scholar could learn.
Several years passed. Elian became one of the most respected scholars in the kingdom.
He studied mathematics, astronomy, philosophy, and history.
Despite all his knowledge, he still wondered about the silver key.
At last the king grew old. Knowing that his time was limited, he summoned Elian to the castle.
Together they walked through long corridors and descended a spiral staircase beneath the royal castle.
At the bottom stood a heavy door.
The king removed the silver key from his cloak and unlocked the door.
Inside was a vast library containing thousands of books collected over many generations.
The king explained that the hidden chamber existed to preserve knowledge during times of war and disaster.
Elian finally understood why the silver key had always been important.
It was not valuable because it was made of silver. 
It was valuable because it protected the wisdom of the kingdom. 
From that day forward Elian became the guardian of the library and the keeper of the silver key."""

chars = sorted(list(set(text)))  # why set then list then sort?

char_to_index = {char: i for i, char in enumerate(chars)}
index_to_char = {i: char for i, char in enumerate(chars)}

### Quick explanation of the building blocks above

- **`set(text)`** — removes duplicate characters, leaving only the *unique* characters (order is not guaranteed).
  ```python
  set("aab")  # {'a', 'b'}
  ```

- **`list(...)`** — converts that set into a list so it can be sorted and indexed.
  ```python
  list({'a', 'b'})  # ['a', 'b']
  ```

- **`sorted(...)`** — sorts the list. For characters this sorts by Unicode code point (e.g. space and uppercase letters come before lowercase letters).
  ```python
  sorted(['b', 'a', ' '])  # [' ', 'a', 'b']
  ```

- **`enumerate(chars)`** — walks through `chars` and pairs each item with its position (index), producing `(0, chars[0]), (1, chars[1]), ...`
  ```python
  list(enumerate(['a', 'b', 'c']))  # [(0, 'a'), (1, 'b'), (2, 'c')]
  ```

- **`char_to_index = {char: i for i, char in enumerate(chars)}`** — a *dict comprehension* that builds a lookup table mapping each character to the integer position it has in `chars`.
  ```python
  chars = [' ', 'G', 'T', 'a']
  char_to_index = {char: i for i, char in enumerate(chars)}
  # {' ': 0, 'G': 1, 'T': 2, 'a': 3}
  ```

This gives every unique character a unique integer ID. `index_to_char` does the reverse mapping (ID → character), which we'll need to turn the model's numeric predictions back into text.

### Why do we do this?

Neural networks only understand **numbers**, not letters. So before we can feed text into our LSTM, we need a way to convert characters ↔ numbers:

- **`char_to_index`** — lets us **encode** the input text into a sequence of integers (numbers) that the model can process.
- **`index_to_char`** — lets us **decode** the model's numeric output (predicted IDs) back into readable characters/text.

In short: encode text → numbers to train the model, decode numbers → text to read what it generates.

### <br>3. Creating Sequences and Labels
To train the LSTM, we need sequences of fixed length (seq_length) and the character following each sequence as the label.

In [ ]:
seq_length = 10
sequences = []
labels = []

for i in range(len(text) - seq_length):
    seq = text[i:i + seq_length]
    label = text[i + seq_length]
    sequences.append([char_to_index[char] for char in seq])
    labels.append(char_to_index[label])

print(sequences[:5])
print(labels[:5])

X = np.array(sequences)
y = np.array(labels)

### What this code does

We're building the **training data** for the LSTM using a sliding window over `text`.

- **`seq_length = 3`** — each input example is 3 characters long, and the model's job is to predict the **next** (4th) character.

- **The loop** slides the window one character at a time across `text`:
  - `seq = text[i:i+seq_length]` → 3 consecutive characters (the **input**)
  - `label = text[i+seq_length]` → the character right after them (the **answer**)

  Example with `text = "This is..."`:
  - `i=0` → `seq = "Thi"`, `label = "s"`
  - `i=1` → `seq = "his"`, `label = " "`
  - `i=2` → `seq = "is "`, `label = "i"`

- **`[char_to_index[char] for char in seq]`** — converts each character in `seq` to its integer ID (e.g. `"Thi"` → `[2, 7, 8]`) and appends it to `sequences`.

- **`char_to_index[label]`** — converts the target character to its integer ID and appends it to `labels`.

- **`X = np.array(sequences)`** / **`y = np.array(labels)`** — convert the Python lists into NumPy arrays:
  - `X` shape: `(num_samples, seq_length)` — the inputs
  - `y` shape: `(num_samples,)` — the correct next character for each input

In short: for every 3-character chunk of text, we record "given these 3 characters, the next one is ___" — that's the pattern the LSTM learns to predict.

### <br>4. Converting Sequences and Labels to One-Hot Encoding
For training we convert X and y into one-hot encoded tensors.

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

X_one_hot = torch.nn.functional.one_hot(X_tensor, num_classes=len(chars)).float()
y_one_hot = torch.nn.functional.one_hot(y_tensor, num_classes=len(chars)).float()

### Why do we need one-hot encoding?

`char_to_index` gives each character a number (e.g. `'a' → 3`), but plain integers imply an **order/magnitude** that doesn't actually exist between characters — the model would wrongly think `'d'` (3) is "more" than `'a'` (0), or that `'b'` is "between" `'a'` and `'c'`.

**One-hot encoding** fixes this by representing each character as a vector of length `len(chars)`, with a `1` at that character's index and `0`s everywhere else — so every character is treated as an equally distinct, independent category.

Example, if `len(chars) = 14` and `char_to_index['a'] = 3`:
```
'a' → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
```

This is also the input/output shape the LSTM expects: at each time step, the input is a vector of size `len(chars)` (one slot per possible character), and the output is a probability distribution over `len(chars)` possible "next characters".

### <br>5. Building the LSTM Model
We create an LSTM model with a hidden layer of 128 units and a Dense output layer with softmax activation. The only structural change from the RNN version is swapping `nn.RNN` for `nn.LSTM`; everything else (one-hot inputs, last-timestep output, linear + softmax head) stays identical.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)  # gates use sigmoid/tanh internally, 1 layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)  # LSTM returns hidden state AND cell state
        out = out[:, -1, :]  # take the output of the last time step
        out = self.fc(out)
        return torch.softmax(out, dim=1)

model = CharLSTM(input_size=len(chars), hidden_size=128, output_size=len(chars))
model

<div align="center">

<table style="border:none; text-align:center; font-family:sans-serif; border-collapse:collapse;">
<tr>
<td></td>
<td style="background:#ffe8b0; border:2px solid #c98a00; border-radius:8px; padding:6px 12px; font-size:14px;"><i>c₁</i></td>
<td></td>
<td style="background:#ffe8b0; border:2px solid #c98a00; border-radius:8px; padding:6px 12px; font-size:14px;"><i>c₂</i></td>
<td></td>
<td style="background:#ffcf73; border:2px solid #c98a00; border-radius:8px; padding:6px 12px; font-size:14px;"><i>c₃</i></td>
<td></td>
</tr>
<tr style="font-size:18px; color:#c98a00;">
<td style="font-size:12px; color:#a85a00; white-space:nowrap;">c₀ (zeros) →</td><td>↑</td><td>→</td><td>↑</td><td>→</td><td>↑</td><td></td>
</tr>
<tr>
<td></td>
<td style="background:#c8f7c5; border:2px solid #2ca02c; border-radius:8px; padding:8px 14px; font-size:15px;"><i>out₁ = h₁</i></td>
<td></td>
<td style="background:#c8f7c5; border:2px solid #2ca02c; border-radius:8px; padding:8px 14px; font-size:15px;"><i>out₂ = h₂</i></td>
<td></td>
<td style="background:#7be07b; border:2px solid #2ca02c; border-radius:8px; padding:8px 14px; font-size:15px;"><i>out₃ = h₃</i></td>
<td></td>
</tr>
<tr style="font-size:22px; color:#2ca02c;">
<td></td><td>↑</td><td></td><td>↑</td><td></td><td>↑</td><td></td>
</tr>
<tr>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">h₀ (zeros) →</td>
<td style="background:#ffd9a0; border:2px solid #e07b00; border-radius:8px; padding:16px 18px; font-size:16px; font-weight:bold;">LSTM<br>cell</td>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">→ h₁ →</td>
<td style="background:#ffd9a0; border:2px solid #e07b00; border-radius:8px; padding:16px 18px; font-size:16px; font-weight:bold;">LSTM<br>cell</td>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">→ h₂ →</td>
<td style="background:#ffd9a0; border:2px solid #e07b00; border-radius:8px; padding:16px 18px; font-size:16px; font-weight:bold;">LSTM<br>cell</td>
<td style="font-size:13px; color:#a85a00; white-space:nowrap;">→ h₃</td>
</tr>
<tr style="font-size:22px; color:#1f77b4;">
<td></td><td>↑</td><td></td><td>↑</td><td></td><td>↑</td><td></td>
</tr>
<tr>
<td></td>
<td style="background:#cfe8ff; border:2px solid #1f77b4; border-radius:8px; padding:8px 14px; font-size:15px;"><i>x₁</i></td>
<td></td>
<td style="background:#cfe8ff; border:2px solid #1f77b4; border-radius:8px; padding:8px 14px; font-size:15px;"><i>x₂</i></td>
<td></td>
<td style="background:#cfe8ff; border:2px solid #1f77b4; border-radius:8px; padding:8px 14px; font-size:15px;"><i>x₃</i></td>
<td></td>
</tr>
</table>

<p style="font-size:12px; color:#c98a00; margin-top:6px;">
<b>top row (orange) = cell state cₜ</b> — the long-term-memory "conveyor belt", only lightly edited at each step.
</p>
<p style="font-size:12px; color:#2ca02c; margin-top:2px;">
<b>out[:, -1, :] = out₃ = h₃</b> → fed into <code>self.fc</code> → <code>softmax</code> → next-character probabilities
</p>

</div>

Think of the LSTM as reading your sequence (e.g. `"Thi"`) **one character at a time**, just like the RNN did — but now it keeps **two** running summaries instead of one:

- **hidden state `h`** — same role as the RNN's summary: what to *output* right now.
- **cell state `c`** — a separate long-term memory "conveyor belt" that gets carried forward mostly unchanged, only lightly edited at each step.

At every time step, the LSTM cell computes **three gates** (each a sigmoid, so values are between 0 and 1 = "how much") plus one candidate update:

```
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)    forget gate  — how much of c_{t-1} to keep
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)    input gate   — how much of the new info to let in
o_t = σ(W_o · [h_{t-1}, x_t] + b_o)    output gate  — how much of c_t to expose as h_t
c̃_t = tanh(W_c · [h_{t-1}, x_t] + b_c)  candidate    — the new info itself

c_t = f_t * c_{t-1} + i_t * c̃_t         new cell state   (forget old + add new)
h_t = o_t * tanh(c_t)                   new hidden state (= out at this step)
```

- **`hidden_size = 128`** — each of `h` and `c` is a vector of 128 numbers, and each gate has its **own** weight matrix — so an LSTM has **~4×** the parameters of a vanilla RNN with the same `hidden_size`.

- **`out, (h_n, c_n) = self.lstm(x)`** — same idea as `nn.RNN`, but now returns a *tuple* `(h_n, c_n)` instead of a single hidden state:
  - `out` = every hidden-state output along the way: `[h1, h2, h3]` (what we use for predictions, same as RNN)
  - `h_n` = the final hidden state, `h3` (already included in `out`)
  - `c_n` = the final **cell state**, `c3` — the RNN has no equivalent of this

- **`out[:, -1, :]`** — same as before: grab `h3`, the summary *after reading the whole sequence*, and feed it to `self.fc`.

- **`batch_first=True`** — unchanged meaning: shapes are `(batch, seq_len, hidden_size)` instead of `(seq_len, batch, hidden_size)`.

**Why this helps:** in a vanilla RNN, `h_t` gets squashed through a `tanh` + matrix-multiply at *every single step*, so gradients flowing back over many steps tend to vanish (or explode). The cell state `c_t` is updated by **addition** (`f_t * c_{t-1} + i_t * c̃_t`), which gives gradients a much more direct path back through time — that's the core trick that lets LSTMs learn longer-range dependencies than plain RNNs.

### <br>6. Compiling and Training the Model
We compile the model using the categorical_crossentropy loss and train it for 500 epochs (lr=0.01) so it has enough iterations to fit this small dataset well.

In [ ]:
epochs = 500
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(epochs):
    optimizer.zero_grad()

    y_pred = model(X_one_hot)
    loss = -(y_one_hot * torch.log(y_pred + 1e-9)).sum(dim=1).mean()  # categorical_crossentropy
    loss.backward()
    optimizer.step()

    accuracy = (y_pred.argmax(dim=1) == y_one_hot.argmax(dim=1)).float().mean()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.4f}")

### What this code does

This is the **training loop** — it repeats `epochs` (500) times, and each pass does:

1. **`optimizer.zero_grad()`** — clear gradients left over from the previous epoch.
2. **`y_pred = model(X_one_hot)`** — run the model on all training sequences to get predicted next-character probabilities.
3. **`loss = ...`** — measure how wrong `y_pred` is compared to the true `y_one_hot` (categorical cross-entropy: lower = better).
4. **`loss.backward()`** — compute gradients (how much each weight contributed to the error).
5. **`optimizer.step()`** — update the model's weights using those gradients (Adam, `lr=0.01`).
6. **`accuracy = ...`** — % of examples where the model's top prediction matches the true next character.
7. Every 50 epochs, print the current loss and accuracy so we can watch the model improve.

In short: predict → measure error → adjust weights → repeat, so the model gradually gets better at guessing the next character.

### <br>7. Generating New Text Using the Trained Model
After training we use a starting sequence to generate new text character by character.

In [ ]:
# Easy
start_seq = "In the ancient kingdom of Arandor there lived"

# Medium
#start_seq = "Several years passed. Elian became"

# Hard (tests long-term memory)
#start_seq = "The king removed the"

generated_text = start_seq

model.eval()
with torch.no_grad():
    for i in range(450):
        x = torch.tensor([[char_to_index[char] for char in generated_text[-seq_length:]]], dtype=torch.long)
        x_one_hot = torch.nn.functional.one_hot(x, num_classes=len(chars)).float()
        prediction = model(x_one_hot)
        next_index = torch.argmax(prediction, dim=1).item()
        next_char = index_to_char[next_index]
        generated_text += next_char

print("Generated Text:")
print(generated_text)

### What this code does

- **`model.eval()`** — switches the model to inference mode (turns off training-only behaviors like dropout/batchnorm updates).
- **`with torch.no_grad():`** — tells PyTorch not to track gradients, since we're only doing predictions, not training. This makes generation faster and uses less memory.
- **The loop (50 times):**
  1. Take the **last `seq_length` characters** of `generated_text` and convert them to integer IDs, then to one-hot vectors (`x_one_hot`) — same encoding used during training.
  2. **`model(x_one_hot)`** — run the model to get predicted probabilities for the next character.
  3. **`torch.argmax(prediction, dim=1)`** — pick the character with the highest probability.
  4. **`index_to_char[next_index]`** — convert that prediction back to a character and append it to `generated_text`.
- After 50 iterations, `generated_text` contains the original seed plus 50 newly generated characters.

In short: repeatedly look at the last 3 characters generated so far, predict the next one, append it, and repeat.